# Skeleton HAR — FedProx Client Sweep (2D + 3D)

Comprehensive training notebook for the **skeleton action recognition** subsystem.

**What this trains:**
- Centralized baseline (upper bound)
- FedProx with **5, 10, 20, 50 clients** (Dirichlet α=0.5)
- Both **2D skeletons** (HRNet COCO-17) and **3D skeletons** (NTU-25 Kinect depth)

**15 classes** with **3-tier severity system**:
- 🔴 **EMERGENCY** (3): falling, staggering, nausea/vomiting
- 🟡 **PAIN** (4): touch head/chest/back/neck
- 🔵 **SYMPTOM** (1): sneeze/cough
- 🟢 **NORMAL** (7): standing, sitting, walking×2, drinking, eating, phone call

**Captures:** per-round accuracy, per-class F1, confusion matrices, per-tier recall, client drift.

**Output:** Best model exported to ONNX for live demo.

**Runtime → T4 GPU** before running (~45 min total).

## 1. Setup

In [ ]:
!pip install -q onnx onnxsim onnxruntime scikit-learn

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'

In [ ]:
import os, pickle, time, json, warnings, copy
from collections import defaultdict, OrderedDict
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = 'cuda'

print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Device: {DEVICE}')

## 2. Configuration

In [ ]:
# ── All hyperparameters ──
CONFIG = {
    # Data
    'clip_len': 100,
    'num_person': 2,
    # Model
    'base_channels': 64,
    'num_stages': 6,
    'dropout': 0.3,
    # Training
    'warmup_epochs': 3,
    'lr': 0.01,
    'momentum': 0.9,
    'weight_decay': 5e-4,
    'batch_size': 64,
    # FL
    'fl_method': 'fedprox',
    'fedprox_mu': 0.01,
    'local_epochs': 1,
    'dirichlet_alpha': 0.5,
    'num_rounds': 50,
    'client_sweep': [5, 10, 20, 50],
    'seed': SEED,
}

NUM_CLASSES = 15

# ── 15-class label set ──
# Grouped by severity tier for the emergency alert system
MEDICAL_LABELS = [
    # EMERGENCY (immediate alert) — indices 0-2
    'falling',                      # 0
    'staggering',                   # 1
    'nausea/vomiting',              # 2
    # PAIN (nurse notify) — indices 3-6
    'touch head (headache)',        # 3
    'touch chest (chest pain)',     # 4
    'touch back (backache)',        # 5
    'touch neck (neckache)',        # 6
    # SYMPTOM (log & monitor) — index 7
    'sneeze/cough',                 # 7
    # NORMAL (no action) — indices 8-14
    'standing up',                  # 8
    'sitting down',                 # 9
    'walking towards',              # 10
    'walking apart',                # 11
    'drink water',                  # 12
    'eat meal',                     # 13
    'phone call',                   # 14
]

# NTU-60 original class indices → our 0–14 labels
MEDICAL_CLASS_MAP = {
    # EMERGENCY
    42: 0,   # falling
    41: 1,   # staggering
    47: 2,   # nausea or vomiting
    # PAIN
    43: 3,   # touch head (headache)
    44: 4,   # touch chest (heart pain)
    45: 5,   # touch back (backache)
    46: 6,   # touch neck (neckache)
    # SYMPTOM
    40: 7,   # sneeze/cough
    # NORMAL
    8:  8,   # standing up
    7:  9,   # sitting down
    58: 10,  # walking towards each other
    59: 11,  # walking apart from each other
    0:  12,  # drink water
    1:  13,  # eat meal/snack
    27: 14,  # make a phone call
}

# ── 3-Tier severity system ──
SEVERITY_TIERS = {
    'EMERGENCY': [0, 1, 2],        # 🔴 immediate alert
    'PAIN':      [3, 4, 5, 6],     # 🟡 nurse notify
    'SYMPTOM':   [7],              # 🔵 log & monitor
    'NORMAL':    [8, 9, 10, 11, 12, 13, 14],  # 🟢 no action
}
TIER_ICONS = {'EMERGENCY': '🔴', 'PAIN': '🟡', 'SYMPTOM': '🔵', 'NORMAL': '🟢'}

# Reverse lookup: class_idx → tier name
CLASS_TO_TIER = {}
for tier, idxs in SEVERITY_TIERS.items():
    for idx in idxs:
        CLASS_TO_TIER[idx] = tier

print('Config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'\n{NUM_CLASSES} classes across {len(SEVERITY_TIERS)} tiers:')
for tier, idxs in SEVERITY_TIERS.items():
    print(f'  {TIER_ICONS[tier]} {tier:10s} ({len(idxs)}): {[MEDICAL_LABELS[i] for i in idxs]}')

## 3. Download NTU-60 Data (2D + 3D)

In [ ]:
data_dir = 'data/nturgbd'
os.makedirs(data_dir, exist_ok=True)

# 2D HRNet skeletons (COCO-17, ~1.1 GB)
pkl_2d = f'{data_dir}/ntu60_hrnet.pkl'
if not os.path.exists(pkl_2d):
    !wget -q --show-progress -O {pkl_2d} \
        https://download.openmmlab.com/mmaction/pyskl/data/nturgbd/ntu60_hrnet.pkl
print(f'2D: {pkl_2d} ({os.path.getsize(pkl_2d)/1e9:.2f} GB)')

# 3D Kinect skeletons (NTU-25, ~0.6 GB)
pkl_3d = f'{data_dir}/ntu60_3danno.pkl'
if not os.path.exists(pkl_3d):
    !wget -q --show-progress -O {pkl_3d} \
        https://download.openmmlab.com/mmaction/pyskl/data/nturgbd/ntu60_3danno.pkl
print(f'3D: {pkl_3d} ({os.path.getsize(pkl_3d)/1e9:.2f} GB)')

In [ ]:
# Load both datasets
with open(pkl_2d, 'rb') as f:
    data_2d = pickle.load(f)
with open(pkl_3d, 'rb') as f:
    data_3d = pickle.load(f)

split_info = data_2d['split']
anns_2d = data_2d['annotations']
anns_3d = data_3d['annotations']

print(f'2D annotations: {len(anns_2d)}')
print(f'3D annotations: {len(anns_3d)}')
print(f'Splits: {list(split_info.keys())}')

In [ ]:
# Filter to 15 medical + context classes
selected = set(MEDICAL_CLASS_MAP.keys())

med_2d = [a for a in anns_2d if a['label'] in selected]
med_3d = [a for a in anns_3d if a['label'] in selected]

print(f'Medical 2D: {len(med_2d)} samples')
print(f'Medical 3D: {len(med_3d)} samples')

# Per-class distribution with severity tier
print(f'\nPer-class counts (2D) — {NUM_CLASSES} classes:')
for orig_cls, new_cls in sorted(MEDICAL_CLASS_MAP.items(), key=lambda x: x[1]):
    count = sum(1 for a in med_2d if a['label'] == orig_cls)
    tier = CLASS_TO_TIER[new_cls]
    icon = TIER_ICONS[tier]
    print(f'  {icon} [{new_cls:2d}] {MEDICAL_LABELS[new_cls]:30s} ({count:4d})  [{tier}]')

# Tier balance summary
print('\nTier balance:')
tier_counts = {}
for tier, idxs in SEVERITY_TIERS.items():
    orig_keys = [k for k, v in MEDICAL_CLASS_MAP.items() if v in idxs]
    tc = sum(1 for a in med_2d if a['label'] in orig_keys)
    tier_counts[tier] = tc
total = sum(tier_counts.values())
for tier, tc in tier_counts.items():
    print(f'  {TIER_ICONS[tier]} {tier:10s}: {tc:5d} ({tc/total:.0%})')
print(f'  {"TOTAL":14s}: {total:5d}')

## 4. STGCN++ Model (Self-Contained, Dual Graph Support)

In [ ]:
# ═══════════════════════════ Graph definitions ════════════════════════════

# COCO-17 (for 2D HRNet skeletons)
COCO_EDGES = [
    (15, 13), (13, 11), (16, 14), (14, 12), (11, 5), (12, 6),
    (9, 7), (7, 5), (10, 8), (8, 6), (5, 0), (6, 0),
    (1, 0), (3, 1), (2, 0), (4, 2),
]
COCO_NUM_JOINTS = 17
COCO_CENTER = 0

# NTU-25 (for 3D Kinect skeletons)
NTU_EDGES = [
    (0, 1), (1, 20), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5),
    (7, 6), (8, 20), (9, 8), (10, 9), (11, 10), (12, 0),
    (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),
    (19, 18), (21, 7), (22, 7), (23, 11), (24, 11),
]
NTU_NUM_JOINTS = 25
NTU_CENTER = 20


def build_adjacency(num_node, edges, center):
    """3-subset spatial adjacency: [identity, inward, outward], row-normalized."""
    def _normalize(mx):
        d = mx.sum(axis=0)
        d[d == 0] = 1
        return mx / d

    I = np.eye(num_node, dtype=np.float32)
    inward = np.zeros((num_node, num_node), dtype=np.float32)
    outward = np.zeros((num_node, num_node), dtype=np.float32)

    # BFS to get hop distances from center
    hop = np.full(num_node, 999, dtype=int)
    hop[center] = 0
    adj_list = {i: [] for i in range(num_node)}
    for u, v in edges:
        adj_list[u].append(v)
        adj_list[v].append(u)
    queue = [center]
    while queue:
        cur = queue.pop(0)
        for nb in adj_list[cur]:
            if hop[nb] > hop[cur] + 1:
                hop[nb] = hop[cur] + 1
                queue.append(nb)

    for u, v in edges:
        if hop[u] < hop[v]:
            inward[v, u] = 1
        else:
            outward[u, v] = 1

    return np.stack([_normalize(I), _normalize(inward), _normalize(outward)])


def get_adjacency(skeleton='coco'):
    if skeleton == 'coco':
        return build_adjacency(COCO_NUM_JOINTS, COCO_EDGES, COCO_CENTER)
    elif skeleton == 'ntu':
        return build_adjacency(NTU_NUM_JOINTS, NTU_EDGES, NTU_CENTER)
    else:
        raise ValueError(f'Unknown skeleton: {skeleton}')

print(f'COCO-17 adjacency: {get_adjacency("coco").shape}')
print(f'NTU-25  adjacency: {get_adjacency("ntu").shape}')

In [ ]:
# ═══════════════════════════ STGCN++ modules ═════════════════════════════

class MsTCN(nn.Module):
    """Multi-scale Temporal Convolution."""
    def __init__(self, in_ch, out_ch, dilations=(1, 2, 3, 4), stride=1, dropout=0.0):
        super().__init__()
        mid = out_ch // (len(dilations) + 2)
        rem = out_ch - mid * (len(dilations) + 2)
        self.branches = nn.ModuleList()
        for d in dilations:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(True),
                nn.Conv2d(mid, mid, (3,1), stride=(stride,1), padding=(d,0), dilation=(d,1)),
                nn.BatchNorm2d(mid)))
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.MaxPool2d((3,1), stride=(stride,1), padding=(1,0)),
            nn.BatchNorm2d(mid)))
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch, mid + rem, 1), nn.BatchNorm2d(mid + rem)))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        outs = [br(x) for br in self.branches]
        min_t = min(o.size(2) for o in outs)
        return self.drop(torch.cat([o[:,:,:min_t,:] for o in outs], dim=1))


class UnitGCN(nn.Module):
    """Graph Convolution with adaptive adjacency."""
    def __init__(self, in_ch, out_ch, A, adaptive=True, with_res=True):
        super().__init__()
        K = A.size(0)
        self.conv = nn.ModuleList([nn.Conv2d(in_ch, out_ch, 1) for _ in range(K)])
        self.register_buffer('A', A)
        self.PA = nn.Parameter(A.clone()) if adaptive else None
        self.bn = nn.BatchNorm2d(out_ch)
        if with_res:
            self.res = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch))
                        if in_ch != out_ch else nn.Identity())
        else:
            self.res = None

    def forward(self, x):
        A = self.A + self.PA if self.PA is not None else self.A
        out = sum(self.conv[k](x @ A[k]) for k in range(A.size(0)))
        out = self.bn(out)
        if self.res is not None:
            out = out + self.res(x)
        return F.relu(out)


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.0):
        super().__init__()
        self.gcn = UnitGCN(in_ch, out_ch, A)
        self.tcn = MsTCN(out_ch, out_ch, stride=stride, dropout=dropout)
        self.relu = nn.ReLU(True)
        if in_ch != out_ch or stride != 1:
            self.res = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch),
                nn.AvgPool2d((stride, 1)) if stride > 1 else nn.Identity())
        else:
            self.res = nn.Identity()

    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.res(x))

print('Modules defined.')

In [ ]:
class STGCNPP(nn.Module):
    """
    STGCN++ — lightweight variant for federated learning.

    Input:  (N, M, T, V, C)  with C=3
    Output: (N, num_classes)

    Works with any skeleton graph (COCO-17 or NTU-25).
    """
    def __init__(self, num_classes=10, in_channels=3, skeleton='coco',
                 base_channels=64, num_stages=6, inflate_stages=[3, 5],
                 down_stages=[3, 5], num_person=2, dropout=0.3):
        super().__init__()
        self.num_person = num_person
        A = torch.from_numpy(get_adjacency(skeleton))

        # Data batch norm
        num_joints = A.shape[1]
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_joints)

        # Build stages
        channels = [in_channels] + [base_channels] * num_stages
        for i in inflate_stages:
            channels[i] *= 2
            for j in range(i + 1, len(channels)):
                channels[j] = channels[i]

        self.stages = nn.ModuleList()
        for i in range(num_stages):
            stride = 2 if (i + 1) in down_stages else 1
            self.stages.append(
                STGCNBlock(channels[i], channels[i + 1], A,
                           stride=stride, dropout=dropout))

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels[-1], num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (N, M, T, V, C)
        N, M, T, V, C = x.shape
        x = x.permute(0, 1, 4, 3, 2).contiguous()  # (N, M, C, V, T)
        x = x.view(N, M * C * V, T)
        x = self.data_bn(x)
        x = x.view(N, M, C, V, T).permute(0, 1, 2, 4, 3)  # (N, M, C, T, V)
        x = x.reshape(N * M, C, T, V)

        for stage in self.stages:
            x = stage(x)

        # Pool: (N*M, C', T', V') → (N*M, C')
        x = self.pool(x).squeeze(-1).squeeze(-1)
        # Merge person dim: (N, M, C') → (N, C') via mean
        x = x.view(N, M, -1).mean(dim=1)
        return self.fc(x)


def build_model(skeleton='coco', **kwargs):
    model = STGCNPP(skeleton=skeleton,
                    num_classes=NUM_CLASSES,
                    base_channels=CONFIG['base_channels'],
                    num_stages=CONFIG['num_stages'],
                    dropout=CONFIG['dropout'],
                    **kwargs).to(DEVICE)
    return model

# Test both
for skel in ['coco', 'ntu']:
    m = build_model(skeleton=skel)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    V = 17 if skel == 'coco' else 25
    dummy = torch.randn(2, 2, 100, V, 3).to(DEVICE)
    out = m(dummy)
    print(f'{skel.upper():5s}: {n_params:,} params, output={out.shape}')
    del m
torch.cuda.empty_cache()

## 5. Preprocessing

In [ ]:
# ════════════════════════ 2D Preprocessing (HRNet) ═══════════════════════

def prenormalize_2d(keypoint, keypoint_score, img_shape=(1080, 1920), threshold=0.01):
    kp = keypoint.astype(np.float32).copy()
    score = keypoint_score.astype(np.float32)
    kp = np.concatenate([kp, score[..., None]], axis=-1)  # (M, T, V, 3)
    mask = kp[..., 2] <= threshold
    h, w = img_shape
    kp[..., 0] = (kp[..., 0] - w / 2) / (w / 2)
    kp[..., 1] = (kp[..., 1] - h / 2) / (h / 2)
    kp[..., 0][mask] = 0
    kp[..., 1][mask] = 0
    return kp

# ════════════════════════ 3D Preprocessing (NTU) ═════════════════════════

def prenormalize_3d(skeleton):
    """PreNormalize3D — center on spine joint, align to body axes."""
    M, T, V, C = skeleton.shape
    if skeleton.sum() == 0:
        return skeleton

    # Find valid frames for main person
    main_person = skeleton[0]
    valid = [t for t in range(T) if not np.allclose(main_person[t], 0)]
    if len(valid) == 0:
        return skeleton

    # Center on spine base (joint 0) at first valid frame
    origin = main_person[valid[0], 0].copy()  # spine base
    skeleton = skeleton - origin[None, None, None, :]

    # Align spine (joint 0→1) to Z-axis
    spine = skeleton[0, valid[0], 1] - skeleton[0, valid[0], 0]
    spine_norm = np.linalg.norm(spine)
    if spine_norm > 1e-6:
        # Scale so spine length = 1
        skeleton = skeleton / spine_norm

    return skeleton.astype(np.float32)

# ══════════════════════ Common utilities ═════════════════════════════════

def uniform_sample(keypoint, clip_len=100):
    M, T, V, C = keypoint.shape
    if T == clip_len:
        return keypoint
    if T < clip_len:
        inds = np.arange(clip_len) % T
    else:
        inds = np.linspace(0, T - 1, clip_len, dtype=int)
    return keypoint[:, inds]

def format_gcn_input(keypoint, num_person=2):
    M, T, V, C = keypoint.shape
    if M < num_person:
        pad = np.zeros((num_person - M, T, V, C), dtype=keypoint.dtype)
        keypoint = np.concatenate([keypoint, pad], axis=0)
    return keypoint[:num_person]

def preprocess_2d(ann, clip_len=100):
    kp = prenormalize_2d(ann['keypoint'], ann['keypoint_score'],
                         ann.get('img_shape', (1080, 1920)))
    kp = uniform_sample(kp, clip_len)
    return format_gcn_input(kp)

def preprocess_3d(ann, clip_len=100):
    kp = prenormalize_3d(ann['keypoint'].astype(np.float32))
    kp = uniform_sample(kp, clip_len)
    return format_gcn_input(kp)

# Test
out_2d = preprocess_2d(med_2d[0])
out_3d = preprocess_3d(med_3d[0])
print(f'2D sample: {out_2d.shape}  (M={out_2d.shape[0]}, T={out_2d.shape[1]}, V={out_2d.shape[2]}, C={out_2d.shape[3]})')
print(f'3D sample: {out_3d.shape}  (M={out_3d.shape[0]}, T={out_3d.shape[1]}, V={out_3d.shape[2]}, C={out_3d.shape[3]})')

## 6. Build Datasets

In [ ]:
%%time

train_dirs = set(split_info['xsub_train'])
test_dirs = set(split_info['xsub_val'])

def build_dataset(annotations, preprocess_fn, name=''):
    """Build train/test numpy arrays from annotations."""
    train_x, train_y, test_x, test_y = [], [], [], []
    for ann in annotations:
        x = preprocess_fn(ann)
        y = MEDICAL_CLASS_MAP[ann['label']]
        if ann['frame_dir'] in train_dirs:
            train_x.append(x); train_y.append(y)
        elif ann['frame_dir'] in test_dirs:
            test_x.append(x); test_y.append(y)
    train_x = np.stack(train_x).astype(np.float32)
    train_y = np.array(train_y, dtype=np.int64)
    test_x = np.stack(test_x).astype(np.float32)
    test_y = np.array(test_y, dtype=np.int64)
    print(f'{name} Train: {train_x.shape}, Test: {test_x.shape}')
    print(f'  Train dist: {np.bincount(train_y, minlength=NUM_CLASSES).tolist()}')
    print(f'  Test  dist: {np.bincount(test_y, minlength=NUM_CLASSES).tolist()}')
    return train_x, train_y, test_x, test_y

# Build both
train_x_2d, train_y_2d, test_x_2d, test_y_2d = build_dataset(med_2d, preprocess_2d, '2D')
print()
train_x_3d, train_y_3d, test_x_3d, test_y_3d = build_dataset(med_3d, preprocess_3d, '3D')

# Free annotation memory
del data_2d, data_3d, anns_2d, anns_3d, med_2d, med_3d
import gc; gc.collect()

## 7. Training & FL Infrastructure

In [ ]:
def make_loader(x, y, batch_size=CONFIG['batch_size'], shuffle=True):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                      batch_size=batch_size, shuffle=shuffle, pin_memory=True)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    all_preds, all_labels = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        preds = model(xb).argmax(1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    return correct / total, np.array(all_preds), np.array(all_labels)


def local_train(model, loader, lr, epochs=1, global_params=None, mu=0.0):
    """Local training with optional FedProx proximal term."""
    opt = torch.optim.SGD(model.parameters(), lr=lr,
                          momentum=CONFIG['momentum'],
                          weight_decay=CONFIG['weight_decay'])
    crit = nn.CrossEntropyLoss()
    model.train()
    total_loss = 0
    n_batches = 0
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if mu > 0 and global_params is not None:
                prox = sum(torch.sum((p - gp.to(DEVICE)) ** 2)
                           for p, gp in zip(model.parameters(), global_params))
                loss = loss + (mu / 2) * prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)


def fedavg_aggregate(global_sd, client_sds, client_sizes):
    """Weighted FedAvg aggregation."""
    total = sum(client_sizes)
    new_sd = OrderedDict()
    for key in global_sd:
        new_sd[key] = sum(
            client_sds[i][key] * (client_sizes[i] / total)
            for i in range(len(client_sds)))
    return new_sd


def calibrate_bn(model, loader, num_batches=20):
    """Recalibrate BatchNorm running stats using global data sample."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
            m.running_mean.zero_()
            m.running_var.fill_(1)
            m.num_batches_tracked.zero_()
    with torch.no_grad():
        for i, (xb, yb) in enumerate(loader):
            if i >= num_batches:
                break
            model(xb.to(DEVICE))
    model.eval()


def dirichlet_split(train_y, n_clients, alpha, seed=42):
    """Label-based Dirichlet non-IID partition."""
    rng = np.random.RandomState(seed)
    label_indices = defaultdict(list)
    for i, label in enumerate(train_y):
        label_indices[int(label)].append(i)

    client_indices = {i: [] for i in range(n_clients)}
    for cls in range(NUM_CLASSES):
        idxs = np.array(label_indices[cls])
        rng.shuffle(idxs)
        proportions = rng.dirichlet([alpha] * n_clients)
        split_points = (np.cumsum(proportions)[:-1] * len(idxs)).astype(int)
        splits = np.split(idxs, split_points)
        for cid, chunk in enumerate(splits):
            client_indices[cid].extend(chunk.tolist())
    return client_indices

print('Training infrastructure ready.')

## 8. Experiment Runner

In [ ]:
def run_experiment(train_x, train_y, test_x, test_y, skeleton='coco',
                   n_clients=5, num_rounds=50, tag=''):
    """
    Full experiment: warmup → federated training → capture all metrics.

    Returns dict with all results and per-round logs.
    """
    mu = CONFIG['fedprox_mu']
    lr = CONFIG['lr']
    label = f'{tag} {skeleton.upper()} {n_clients}C'
    print(f'\n{"="*60}')
    print(f'  {label} — FedProx (μ={mu}), {num_rounds}R, α={CONFIG["dirichlet_alpha"]}')
    print(f'{"="*60}')

    # ── Dirichlet split ──
    client_idx = dirichlet_split(train_y, n_clients, CONFIG['dirichlet_alpha'])
    client_sizes = [len(client_idx[c]) for c in range(n_clients)]
    print(f'Client sizes: {client_sizes}')
    for cid in range(min(n_clients, 10)):  # print up to 10
        dist = np.bincount(train_y[client_idx[cid]], minlength=NUM_CLASSES)
        print(f'  C{cid}: {len(client_idx[cid]):4d} samples, dist={dist.tolist()}')
    if n_clients > 10:
        print(f'  ... ({n_clients - 10} more clients)')

    # ── Build loaders ──
    client_loaders = {}
    for cid in range(n_clients):
        ci = client_idx[cid]
        client_loaders[cid] = make_loader(train_x[ci], train_y[ci])
    test_loader = make_loader(test_x, test_y, shuffle=False)
    # Small calibration loader from train data
    calib_loader = make_loader(train_x, train_y, batch_size=64, shuffle=True)

    # ── Build model ──
    global_model = build_model(skeleton=skeleton)
    n_params = sum(p.numel() for p in global_model.parameters() if p.requires_grad)
    print(f'Model: {n_params:,} params')

    # ── Warmup (centralized pretraining) ──
    print(f'\nWarmup: {CONFIG["warmup_epochs"]} epochs centralized...')
    warmup_loader = make_loader(train_x, train_y)
    for ep in range(CONFIG['warmup_epochs']):
        loss = local_train(global_model, warmup_loader, lr=lr, epochs=1)
        acc, _, _ = evaluate(global_model, test_loader)
        print(f'  Warmup {ep+1}: loss={loss:.4f}, test_acc={acc:.1%}')
    warmup_state = copy.deepcopy(global_model.state_dict())

    # ── Federated rounds ──
    logs = {
        'round': [], 'global_acc': [], 'global_f1': [],
        'mean_client_acc': [], 'worst_client_acc': [],
        'mean_loss': [],
    }
    best_acc = 0
    best_state = None

    t0 = time.time()
    for rnd in range(1, num_rounds + 1):
        global_params = [p.clone().detach().cpu() for p in global_model.parameters()]
        global_sd = copy.deepcopy(global_model.state_dict())

        client_sds = []
        client_losses = []
        client_accs = []

        for cid in range(n_clients):
            local = build_model(skeleton=skeleton)
            local.load_state_dict(global_sd)
            loss = local_train(local, client_loaders[cid], lr=lr,
                               epochs=CONFIG['local_epochs'],
                               global_params=global_params, mu=mu)
            client_sds.append(OrderedDict({k: v.cpu() for k, v in local.state_dict().items()}))
            client_losses.append(loss)

            c_acc, _, _ = evaluate(local, client_loaders[cid])
            client_accs.append(c_acc)
            del local

        # Aggregate
        new_sd = fedavg_aggregate(global_sd, client_sds, client_sizes)
        global_model.load_state_dict(new_sd)

        # BN calibration
        calibrate_bn(global_model, calib_loader)

        # Evaluate
        g_acc, g_preds, g_labels = evaluate(global_model, test_loader)
        g_f1 = f1_score(g_labels, g_preds, average='macro')

        logs['round'].append(rnd)
        logs['global_acc'].append(g_acc)
        logs['global_f1'].append(g_f1)
        logs['mean_client_acc'].append(np.mean(client_accs))
        logs['worst_client_acc'].append(np.min(client_accs))
        logs['mean_loss'].append(np.mean(client_losses))

        if g_acc > best_acc:
            best_acc = g_acc
            best_state = copy.deepcopy(global_model.state_dict())

        if rnd % 5 == 0 or rnd == 1:
            elapsed = time.time() - t0
            print(f'  R{rnd:3d}: acc={g_acc:.1%}  f1={g_f1:.3f}  '
                  f'client_mean={np.mean(client_accs):.1%}  '
                  f'worst={np.min(client_accs):.1%}  [{elapsed:.0f}s]')

    elapsed = time.time() - t0
    print(f'\nBest accuracy: {best_acc:.1%} ({elapsed:.0f}s total)')

    # Final eval with best state
    global_model.load_state_dict(best_state)
    calibrate_bn(global_model, calib_loader)
    final_acc, final_preds, final_labels = evaluate(global_model, test_loader)

    return {
        'tag': label,
        'skeleton': skeleton,
        'n_clients': n_clients,
        'best_acc': best_acc,
        'final_acc': final_acc,
        'final_preds': final_preds,
        'final_labels': final_labels,
        'best_state': best_state,
        'warmup_state': warmup_state,
        'logs': logs,
        'n_params': n_params,
        'elapsed': elapsed,
        'client_sizes': client_sizes,
    }

print('Experiment runner ready.')

## 9. Centralized Baselines

In [ ]:
%%time

def train_centralized(train_x, train_y, test_x, test_y, skeleton='coco',
                      epochs=50, tag=''):
    label = f'Centralized {skeleton.upper()}'
    print(f'\n{"="*60}')
    print(f'  {label} — {epochs} epochs')
    print(f'{"="*60}')

    model = build_model(skeleton=skeleton)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model: {n_params:,} params')

    train_loader = make_loader(train_x, train_y)
    test_loader = make_loader(test_x, test_y, shuffle=False)

    opt = torch.optim.SGD(model.parameters(), lr=CONFIG['lr'],
                          momentum=CONFIG['momentum'],
                          weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    history = {'epoch': [], 'train_acc': [], 'test_acc': [], 'loss': []}
    best_acc = 0
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        ep_loss = 0
        n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item() * xb.size(0)
            n += xb.size(0)
        scheduler.step()

        train_acc, _, _ = evaluate(model, train_loader)
        test_acc, _, _ = evaluate(model, test_loader)
        history['epoch'].append(ep)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['loss'].append(ep_loss / n)

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())

        if ep % 10 == 0 or ep == 1:
            print(f'  Ep {ep:3d}: loss={ep_loss/n:.4f}  train={train_acc:.1%}  test={test_acc:.1%}')

    model.load_state_dict(best_state)
    final_acc, preds, labels = evaluate(model, test_loader)
    print(f'Best test accuracy: {best_acc:.1%}')

    return {
        'tag': label, 'skeleton': skeleton, 'n_clients': 0,
        'best_acc': best_acc, 'final_acc': final_acc,
        'final_preds': preds, 'final_labels': labels,
        'best_state': best_state, 'logs': history,
        'n_params': n_params,
    }

# Train centralized baselines
results = {}
results['cent_2d'] = train_centralized(train_x_2d, train_y_2d, test_x_2d, test_y_2d,
                                        skeleton='coco', epochs=50)
results['cent_3d'] = train_centralized(train_x_3d, train_y_3d, test_x_3d, test_y_3d,
                                        skeleton='ntu', epochs=50)

## 10. FedProx Client Sweep — 2D (COCO-17)

In [ ]:
%%time

for nc in CONFIG['client_sweep']:
    key = f'fedprox_2d_{nc}c'
    results[key] = run_experiment(
        train_x_2d, train_y_2d, test_x_2d, test_y_2d,
        skeleton='coco', n_clients=nc,
        num_rounds=CONFIG['num_rounds'], tag='FedProx')
    torch.cuda.empty_cache()

# Quick summary
print('\n' + '='*50)
print('2D SWEEP SUMMARY')
print('='*50)
print(f'{"Centralized":20s}: {results["cent_2d"]["best_acc"]:.1%}')
for nc in CONFIG['client_sweep']:
    key = f'fedprox_2d_{nc}c'
    print(f'{"FedProx " + str(nc) + "C":20s}: {results[key]["best_acc"]:.1%}')

## 11. FedProx — 3D (NTU-25) with 10 Clients

15 classes, same tier system. Direct comparison with 2D to evaluate depth camera feasibility.

In [ ]:
%%time

# 3D: run with 10 clients to compare directly with 2D
results['fedprox_3d_10c'] = run_experiment(
    train_x_3d, train_y_3d, test_x_3d, test_y_3d,
    skeleton='ntu', n_clients=10,
    num_rounds=CONFIG['num_rounds'], tag='FedProx 3D')

print(f'\n3D centralized: {results["cent_3d"]["best_acc"]:.1%}')
print(f'3D FedProx 10C: {results["fedprox_3d_10c"]["best_acc"]:.1%}')

## 12. Results Summary

In [ ]:
print('='*65)
print(f'{"Method":30s} {"Skeleton":10s} {"Clients":>8s} {"Best Acc":>10s}')
print('-'*65)

# Print ordered
order = ['cent_2d', 'cent_3d'] + \
        [f'fedprox_2d_{nc}c' for nc in CONFIG['client_sweep']] + \
        ['fedprox_3d_10c']

for key in order:
    if key not in results:
        continue
    r = results[key]
    nc = str(r['n_clients']) if r['n_clients'] > 0 else 'all'
    print(f'{r["tag"]:30s} {r["skeleton"]:10s} {nc:>8s} {r["best_acc"]:>9.1%}')

print('='*65)

# Per-class F1 for best 2D result
best_2d_key = min(
    [k for k in results if '2d' in k],
    key=lambda k: -results[k]['best_acc'])
r = results[best_2d_key]
print(f'\nBest 2D: {r["tag"]} ({r["best_acc"]:.1%})')
print(classification_report(r['final_labels'], r['final_preds'],
                            target_names=MEDICAL_LABELS, digits=3))

# ── Per-tier recall (critical metric for emergency system) ──
def tier_recall(labels, preds):
    """Compute recall per severity tier."""
    print('\nPer-tier recall:')
    for tier, idxs in SEVERITY_TIERS.items():
        tier_mask = np.isin(labels, idxs)
        if tier_mask.sum() == 0:
            continue
        tier_correct = (preds[tier_mask] == labels[tier_mask]).sum()
        tier_total = tier_mask.sum()
        recall = tier_correct / tier_total
        icon = TIER_ICONS[tier]
        print(f'  {icon} {tier:10s}: {recall:.1%} ({tier_correct}/{tier_total})')

tier_recall(r['final_labels'], r['final_preds'])

## 13. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Skeleton HAR — FedProx Client Sweep (2D COCO-17)',
             fontsize=14, fontweight='bold')

# ── Panel 1: Global accuracy over rounds ──
ax = axes[0]
colors = ['#1565c0', '#43a047', '#e65100', '#6a1b9a']
for i, nc in enumerate(CONFIG['client_sweep']):
    key = f'fedprox_2d_{nc}c'
    if key in results:
        ax.plot(results[key]['logs']['global_acc'],
                label=f'{nc} clients', color=colors[i], linewidth=2)
ax.axhline(results['cent_2d']['best_acc'], ls='--', color='black',
           label='Centralized', linewidth=1.5)
ax.set_xlabel('Round')
ax.set_ylabel('Global Test Accuracy')
ax.set_title('Accuracy vs Round')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Panel 2: Macro F1 over rounds ──
ax = axes[1]
for i, nc in enumerate(CONFIG['client_sweep']):
    key = f'fedprox_2d_{nc}c'
    if key in results:
        ax.plot(results[key]['logs']['global_f1'],
                label=f'{nc} clients', color=colors[i], linewidth=2)
ax.set_xlabel('Round')
ax.set_ylabel('Macro F1')
ax.set_title('F1 Score vs Round')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Panel 3: Bar chart comparison ──
ax = axes[2]
methods = ['Centralized'] + [f'{nc}C' for nc in CONFIG['client_sweep']]
accs = [results['cent_2d']['best_acc']] + \
       [results[f'fedprox_2d_{nc}c']['best_acc'] for nc in CONFIG['client_sweep']]
bar_colors = ['#424242'] + colors
bars = ax.bar(methods, accs, color=bar_colors)
ax.set_ylabel('Best Test Accuracy')
ax.set_title('Peak Accuracy Comparison')
ax.set_ylim(0, 1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.1%}',
            ha='center', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sweep_2d_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Worst-client accuracy (fairness metric) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Client Fairness & 2D vs 3D', fontsize=14, fontweight='bold')

ax = axes[0]
for i, nc in enumerate(CONFIG['client_sweep']):
    key = f'fedprox_2d_{nc}c'
    if key in results:
        ax.plot(results[key]['logs']['worst_client_acc'],
                label=f'{nc}C worst', color=colors[i], linewidth=2)
        ax.plot(results[key]['logs']['mean_client_acc'],
                label=f'{nc}C mean', color=colors[i], linewidth=1, ls='--', alpha=0.6)
ax.set_xlabel('Round')
ax.set_ylabel('Accuracy')
ax.set_title('Client Fairness (worst vs mean)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# 2D vs 3D comparison
ax = axes[1]
labels_bar = ['Cent 2D', 'FP-10C 2D', 'Cent 3D', 'FP-10C 3D']
accs_bar = [
    results['cent_2d']['best_acc'],
    results.get('fedprox_2d_10c', {}).get('best_acc', 0),
    results['cent_3d']['best_acc'],
    results.get('fedprox_3d_10c', {}).get('best_acc', 0),
]
bar_c = ['#1565c0', '#42a5f5', '#c62828', '#ef5350']
bars = ax.bar(labels_bar, accs_bar, color=bar_c)
for bar, acc in zip(bars, accs_bar):
    if acc > 0:
        ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.1%}',
                ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Best Test Accuracy')
ax.set_title('2D (HRNet) vs 3D (Kinect) Skeleton')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sweep_fairness_2d_vs_3d.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ──
show_keys = ['cent_2d', 'fedprox_2d_10c', 'cent_3d', 'fedprox_3d_10c']
show_keys = [k for k in show_keys if k in results]
n = len(show_keys)

fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for i, key in enumerate(show_keys):
    ax = axes[i]
    r = results[key]
    cm = confusion_matrix(r['final_labels'], r['final_preds'])
    ax.imshow(cm, cmap='Blues')
    short = [l.split('(')[0].strip()[:8] for l in MEDICAL_LABELS]
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(short, rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(short, fontsize=6)
    ax.set_title(f'{r["tag"]}\n{r["best_acc"]:.1%}', fontweight='bold', fontsize=10)
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, str(cm[row, col]), ha='center', va='center',
                    fontsize=6, color='w' if cm[row, col] > cm.max()*0.5 else 'k')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Export Best Model to ONNX

In [ ]:
import onnx

os.makedirs('outputs', exist_ok=True)

def export_onnx(state_dict, skeleton, filename):
    model = build_model(skeleton=skeleton)
    model.load_state_dict(state_dict)
    model.eval()

    V = 17 if skeleton == 'coco' else 25
    dummy = torch.randn(1, 2, 100, V, 3).to(DEVICE)

    torch.onnx.export(
        model, dummy, filename,
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=17)

    # Verify
    onnx_model = onnx.load(filename)
    onnx.checker.check_model(onnx_model)
    size_mb = os.path.getsize(filename) / 1e6
    print(f'Exported: {filename} ({size_mb:.1f} MB)')

    # Simplify
    try:
        from onnxsim import simplify
        simplified, ok = simplify(onnx_model)
        if ok:
            onnx.save(simplified, filename)
            new_size = os.path.getsize(filename) / 1e6
            print(f'Simplified: {new_size:.1f} MB')
    except Exception as e:
        print(f'Simplification skipped: {e}')

# Export best 2D model (for live demo with webcam)
best_2d_key = min([k for k in results if '2d' in k],
                  key=lambda k: -results[k]['best_acc'])
export_onnx(results[best_2d_key]['best_state'], 'coco',
            'outputs/stgcnpp_medical_2d.onnx')

# Export best 3D model
if 'cent_3d' in results:
    best_3d_key = min([k for k in results if '3d' in k],
                      key=lambda k: -results[k]['best_acc'])
    export_onnx(results[best_3d_key]['best_state'], 'ntu',
                'outputs/stgcnpp_medical_3d.onnx')

# Also save PyTorch checkpoints
for key, r in results.items():
    if 'best_state' in r and r.get('best_state') is not None:
        torch.save(r['best_state'], f'outputs/{key}_best.pt')
print('\nAll checkpoints saved to outputs/')

In [ ]:
# Verify ONNX inference
import onnxruntime as ort

for onnx_file, skel in [('outputs/stgcnpp_medical_2d.onnx', 'coco'),
                         ('outputs/stgcnpp_medical_3d.onnx', 'ntu')]:
    if not os.path.exists(onnx_file):
        continue
    sess = ort.InferenceSession(onnx_file)
    V = 17 if skel == 'coco' else 25
    dummy = np.random.randn(1, 2, 100, V, 3).astype(np.float32)
    out = sess.run(None, {'input': dummy})[0]
    print(f'{onnx_file}: input=(1,2,100,{V},3) → output={out.shape} ✓')

## 15. Save Full Results & Label Map

In [ ]:
# Save results JSON (all metrics, no tensors)
results_json = {}
for key, r in results.items():
    entry = {
        'tag': r['tag'],
        'skeleton': r['skeleton'],
        'n_clients': r['n_clients'],
        'best_acc': float(r['best_acc']),
        'n_params': r.get('n_params', 0),
    }
    # Include per-round logs
    if 'logs' in r:
        logs = r['logs']
        entry['logs'] = {k: [float(v) for v in vals] for k, vals in logs.items()
                         if isinstance(vals, list) and len(vals) > 0
                         and isinstance(vals[0], (int, float, np.floating))}
    results_json[key] = entry

results_json['config'] = CONFIG
results_json['severity_tiers'] = {k: v for k, v in SEVERITY_TIERS.items()}

with open('outputs/results_sweep.json', 'w') as f:
    json.dump(results_json, f, indent=2, default=str)
print('Saved: outputs/results_sweep.json')

# Save label map (for inference demo) — includes tier metadata
with open('outputs/label_map_medical_15.txt', 'w') as f:
    for i, label in enumerate(MEDICAL_LABELS):
        tier = CLASS_TO_TIER[i]
        f.write(f'{label}\n')
print('Saved: outputs/label_map_medical_15.txt')

# Save tier mapping (for demo alert system)
tier_map = {
    'labels': MEDICAL_LABELS,
    'tiers': SEVERITY_TIERS,
    'class_to_tier': {str(k): v for k, v in CLASS_TO_TIER.items()},
    'tier_colors': {'EMERGENCY': '#FF0000', 'PAIN': '#FFA500',
                    'SYMPTOM': '#4488FF', 'NORMAL': '#44DD44'},
}
with open('outputs/tier_map.json', 'w') as f:
    json.dump(tier_map, f, indent=2)
print('Saved: outputs/tier_map.json')

# Print what to download
print('\n📥 Download these files for demo:')
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}')
    print(f'  outputs/{f}  ({size/1e6:.1f} MB)' if size > 1e6
          else f'  outputs/{f}  ({size/1e3:.0f} KB)')

In [ ]:
# ── Optional: Copy to Google Drive ──
# DRIVE_OUT = '/content/drive/MyDrive/pyskl_results'
# os.makedirs(DRIVE_OUT, exist_ok=True)
# !cp outputs/* "{DRIVE_OUT}/"
# !cp *.png "{DRIVE_OUT}/"
# print(f'Copied to {DRIVE_OUT}')

## 16. Summary & Next Steps

### 15-Class 3-Tier Alert System

| Tier | Color | Classes | Alert Action |
|------|-------|---------|-------------|
| 🔴 EMERGENCY | Red | falling, staggering, nausea/vomiting | Immediate alert to nurse station |
| 🟡 PAIN | Yellow | touch head/chest/back/neck | Nurse notification within minutes |
| 🔵 SYMPTOM | Blue | sneeze/cough | Logged for review |
| 🟢 NORMAL | Green | stand, sit, walk×2, drink, eat, phone | No action |

### Results Table

| Method | Skeleton | Clients | Best Acc |
|--------|----------|---------|----------|
| Centralized | 2D (COCO-17) | — | TBD |
| Centralized | 3D (NTU-25) | — | TBD |
| FedProx | 2D | 5 | TBD |
| FedProx | 2D | 10 | TBD |
| FedProx | 2D | 20 | TBD |
| FedProx | 2D | 50 | TBD |
| FedProx | 3D | 10 | TBD |

### Key Questions Answered
1. **How does FedProx scale with more clients?** → Client sweep shows degradation curve
2. **Is 3D skeleton better than 2D?** → Direct comparison (same model, same classes)
3. **Does the model distinguish tiers reliably?** → Per-tier recall shows EMERGENCY detection rate
4. **Can the model handle hard negatives?** → phone call vs headache, drink water vs nausea

### Files for Demo
- `stgcnpp_medical_2d.onnx` — for webcam demo (YOLOX → RTMPose → this)
- `stgcnpp_medical_3d.onnx` — for depth camera demo (if 3D proves better)
- `label_map_medical_15.txt` — 15 class labels
- `tier_map.json` — severity tier definitions + colors for demo overlay
- `results_sweep.json` — all metrics for slides